In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import seaborn as sns
import matplotlib.pyplot as plt
import re


In [17]:
df = pd.read_csv("/home/rashid/Documents/FYP/Ai-Augumented-IDS/data/raw/csic_database.csv")

print("Original columns before any fix:")
print(df.columns.tolist())


df = df.rename(columns={df.columns[0]: 'label'})

print("\nFixed column names:")
print(df.columns.tolist())

print("\nDataset shape:", df.shape)

print("\n=== Label Distribution ===")
print(df['label'].value_counts())
print("\nNormalized:")
print(df['label'].value_counts(normalize=True).round(4))

print("\nFirst 5 rows preview:")
print(df.head())

Original columns before any fix:
['Unnamed: 0', 'Method', 'User-Agent', 'Pragma', 'Cache-Control', 'Accept', 'Accept-encoding', 'Accept-charset', 'language', 'host', 'cookie', 'content-type', 'connection', 'lenght', 'content', 'classification', 'URL']

Fixed column names:
['label', 'Method', 'User-Agent', 'Pragma', 'Cache-Control', 'Accept', 'Accept-encoding', 'Accept-charset', 'language', 'host', 'cookie', 'content-type', 'connection', 'lenght', 'content', 'classification', 'URL']

Dataset shape: (61065, 17)

=== Label Distribution ===
label
Normal       36000
Anomalous    25065
Name: count, dtype: int64

Normalized:
label
Normal       0.5895
Anomalous    0.4105
Name: proportion, dtype: float64

First 5 rows preview:
    label Method                                         User-Agent    Pragma  \
0  Normal    GET  Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...  no-cache   
1  Normal    GET  Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...  no-cache   
2  Normal   POST  Mozilla/5.0

In [18]:

def extract_security_features(df):
    df = df.copy()
    
    df['payload'] = df['URL'].fillna('').astype(str)
    if 'content' in df.columns:
        df['payload'] = df['payload'] + " " + df['content'].fillna('').astype(str)
    
    df['payload'] = df['payload'].str.strip()
    
    print(f"Payload created successfully!")
    print(f"Average payload length: {df['payload'].str.len().mean():.1f} characters")
    
    df['payload_length'] = df['payload'].str.len()
    

    df['special_chars'] = df['payload'].apply(
        lambda x: len(re.findall(r"[!@#$%^&*()_+=\[\]{};:'\"\\|,.<>/?%\-]", x))
    )
    df['special_ratio'] = np.where(df['payload_length'] > 0,
                                   df['special_chars'] / df['payload_length'], 0)
    
    
    df['percent_count'] = df['payload'].str.count('%')       
    df['percent_ratio'] = np.where(df['payload_length'] > 0, 
                                   df['percent_count'] / df['payload_length'], 0)
    
    df['sql_keywords'] = df['payload'].str.lower().str.count(r'select|union|and |or |1=1|--|drop|insert')
    df['semicolon_count'] = df['payload'].str.count(';')
    df['plus_count'] = df['payload'].str.count(r'\+')
    
    df['slash_count'] = df['payload'].str.count('/')
    df['dot_count'] = df['payload'].str.count(r'\.')
    df['param_count'] = df['payload'].str.count('=')
    
    
    df['upper_ratio'] = df['payload'].apply(
        lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
    )
    df['digit_ratio'] = df['payload'].apply(
        lambda x: sum(1 for c in x if c.isdigit()) / len(x) if len(x) > 0 else 0
    )
    
    return df



df = extract_security_features(df)

feature_cols = [
    'payload_length', 'special_chars', 'special_ratio',
    'percent_count', 'percent_ratio', 'sql_keywords',
    'semicolon_count', 'plus_count', 'slash_count', 
    'dot_count', 'param_count', 'upper_ratio', 'digit_ratio'
]

print("\nFeatures created successfully!")
print(df[feature_cols].describe().round(3))

Payload created successfully!
Average payload length: 122.6 characters

Features created successfully!
       payload_length  special_chars  special_ratio  percent_count  \
count       61065.000      61065.000      61065.000      61065.000   
mean          122.557         20.057          0.169          1.884   
std            97.069         15.056          0.019          6.443   
min            31.000          6.000          0.115          0.000   
25%            58.000         10.000          0.155          0.000   
50%            76.000         12.000          0.169          0.000   
75%           130.000         24.000          0.179          2.000   
max           895.000        163.000          0.262        114.000   

       percent_ratio  sql_keywords  semicolon_count  plus_count  slash_count  \
count      61065.000     61065.000          61065.0   61065.000    61065.000   
mean           0.008         0.124              0.0       1.427        6.015   
std            0.018      

In [ ]:

print("Available columns that might be the label:")
print([col for col in df.columns if 'class' in col.lower() or 'label' in col.lower()])


if 'classification' in df.columns:
    label_col = 'classification'
elif 'label' in df.columns:
    label_col = 'label'
else:
    raise ValueError("Could not find label/classification column!")


y = (df[label_col].astype(str).str.lower().str.contains('anomalous|attack|malicious', na=False)).astype(int)

print("\nBinary label distribution (0 = Normal, 1 = Attack):")
print(y.value_counts())
print(y.value_counts(normalize=True).round(4))


X = df[feature_cols]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25, 
    stratify=y, 
    random_state=42
)

print("\nTrain set shape:", X_train.shape)
print("Test set shape: ", X_test.shape)
print("Attack ratio in train:", y_train.mean().round(4))
print("Attack ratio in test: ", y_test.mean().round(4))

Available columns that might be the label:
['label', 'classification']

Binary label distribution (0 = Normal, 1 = Attack):
classification
0    61065
Name: count, dtype: int64
classification
0    1.0
Name: proportion, dtype: float64

Train set shape: (45798, 13)
Test set shape:  (15267, 13)
Attack ratio in train: 0.0
Attack ratio in test:  0.0
